# Week 3 — Solution (v4)

End-to-end honest closed-loop control of the `GG4.Brain`. The arc, per the v4 spec:

1. **PROBE** — single noisy probe of the real Brain (`T_cal = 1000`).
2. **IDENTIFY** — honest pipeline: N4SID → affine EM at `n = 6`.
3. **BENCHMARK** — basis-free invariants vs the quarantined ground-truth yardstick.
4. **READOUTS** — both 1-D dominant-controllable and 2-D PCA, as config.
5. **CONTROL** — closed-loop hold / track / suppress on the real Brain.
6. **EVALUATE** — settling (band-occupancy) and steady-state error separated, anchored to the per-readout noise floor.
7. **LIMITATIONS** — predicted findings (suppress = OL by physics; tracking buried at noise floor) + the three-way variance decomposition.

**Boundaries kept throughout:** the honest pipeline never touches the yardstick. The yardstick lives in `analysis/`, is used only for *target anchoring* and *invariant comparison*, and the deployed controller never reads it.

All findings, milestone-by-milestone numbers, and the full RESULTS log live in [`results/RESULTS.md`](results/RESULTS.md).

## Setup

If you haven't yet installed the `GG4` wheel (which is the simulator package), the next cell installs the one matching your Python + platform from `wheels/`.

In [ ]:
import sys, platform, subprocess
from pathlib import Path

ROOT = Path('.').resolve()
WHEEL_ROOT = ROOT / 'wheels'
if WHEEL_ROOT.exists():
    py_tag = f'cp{sys.version_info[0]}{sys.version_info[1]}'
    system = platform.system(); machine = platform.machine().lower()
    patterns = []
    if system == 'Windows':
        patterns = [f'*{py_tag}*win_amd64.whl']
    elif system == 'Darwin':
        patterns = [f'*{py_tag}*macosx*universal2.whl', f'*{py_tag}*macosx*arm64.whl']
    elif system == 'Linux':
        patterns = [f'*{py_tag}*manylinux*x86_64.whl']
    wheel = None
    for pat in patterns:
        matches = sorted(WHEEL_ROOT.rglob(pat))
        if matches:
            wheel = matches[0]; break
    if wheel is not None:
        try:
            import GG4  # noqa: F401
            print('GG4 already installed.')
        except ImportError:
            print(f'Installing {wheel.name} ...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel)])
            print('done.')
    else:
        print(f'No wheel matching {py_tag} found in {WHEEL_ROOT}; skipping install.')
else:
    print('No wheels/ directory; assuming GG4 is on PYTHONPATH.')

## Run the full arc

`run_week3.py` is the script form of this notebook. We invoke its top-level functions directly so the output is inline. The arc takes ~100 s on a laptop.

In [ ]:
import sys; sys.path.insert(0, str(Path('.').resolve()))
sys.path.insert(0, str(Path('..').resolve() / 'week 1'))

import run_week3

SEED = 0           # the Brain seed for the headline arc
MULTI_SEED = 5     # number of Brain seeds for the multi-seed sweep
T_CAL, T_RUN = 1000, 300

arc = run_week3.arc_single_seed(seed=SEED, T_cal=T_CAL, T_run=T_RUN)

**Reading the headline numbers above:**

- `ss × NF` is steady-state error in multiples of the per-readout noise floor. **1.0 = at the noise floor.** OpenLoop's ~11.7 means it drifts to ~11× the noise scale.
- `occ` (band occupancy) is the fraction of the final 50 % of steps inside the ±10 % band. Decoupled from steady-state error by design (M3/M4 metric refactor).
- `sat` is the input-saturation fraction.

**Why does PropFB / PI lose to LQG?** They are *model-aware static feedback* (`K_p = α · pinv(G_readout)`, threshold-regularised). On this near-rank-1 system, static feedback cannot compete with LQG's *dynamic optimal* feedback through the Kalman filter — *that* is the finding.

**No purely model-free controller is viable on this system.** Rank-1 input-output geometry means you must know `G`'s direction (and exclude its null space) to push the right way. We state the finding rather than ship a model-free baseline that secretly uses `G`.

## Multi-seed validation

The ~9× closed-loop win replicates across 5 independent Brain seeds (each gives the same `(A, B, C)` per M1, but a different noise realisation).

In [ ]:
multi = run_week3.arc_multi_seed(seeds=list(range(MULTI_SEED)),
                                  T_cal=T_CAL, T_run=T_RUN)

## Eval-noise spread — demonstrating the variance decomposition

M1 proved `(A, B, C)` are identical across Brain seeds to ~1e-14. M5 showed the closed-loop hold metric varies seed-to-seed by ~22 %. So that variation cannot be Brain structural — it must be the eval-time noise stream. The next cell demonstrates this directly: take *one* calibration (Brain seed 0, probe seed 42) and run the resulting LQG on Brain seeds 0…4 — each gives an identical model in invariants but a different eval-time noise stream.

In [ ]:
eval_noise = run_week3.eval_noise_spread(cal_seed=SEED, T_cal=T_CAL, T_run=T_RUN,
                                          eval_seeds=tuple(range(MULTI_SEED)))
print(f'\nss × NF across {MULTI_SEED} eval seeds with one fixed calibration:')
for r in eval_noise['results']:
    print(f"  eval seed {r['eval_seed']}: ss × NF = {r['ss_err_x_noise']:.2f}")
spread = max(r['ss_err_x_noise'] for r in eval_noise['results']) - \
         min(r['ss_err_x_noise'] for r in eval_noise['results'])
print(f"\nspread = {spread:.2f}  (Brain is fixed; only the noise stream varies)")

## Save the headline figure

In [ ]:
out_path = Path('results') / 'week3_solution.png'
out_path.parent.mkdir(parents=True, exist_ok=True)
run_week3.plot_solution(arc, multi, eval_noise, out_path)
print(f'figure saved: {out_path}')
from IPython.display import Image, display
display(Image(filename=str(out_path)))

## Findings — writeup-ready summary

**Headline.** The honest pipeline (N4SID → EM at `n = 6`, from one noisy probe) identifies a model that *predicts* at the noise floor and *controls* within ~10 % of the structurally-correct yardstick, achieving a **~9× closed-loop win over open-loop** on the 2-D PCA readout across 5 Brain seeds.

**The finding with teeth.** Good held-out prediction is **necessary but not sufficient** for good control. On a sloppy, near-rank-1 system, prediction RMS is mostly blind to the controllable σ₀ direction (M3 seed-1 × output_ssi counter-example: prediction at 1.06 but control 5× worse, because σ₀(fit) was 4× too small). **Grade models by the σ₀(G) ratio and closed-loop performance, not by prediction RMS alone.**

**Variance decomposition (M5/M6).**
- Brain seed-invariant (M1: 1e-14)
- Calibration luck ±4–7 % (M5 probe-variance sub-study)
- Eval-time noise ~10–15 % (M6 eval-noise repetition, demonstrated directly)

**Predicted findings (NOT bugs).**
- SUPPRESS @ `z₀` = OpenLoop wins by physics (one-sided actuator can't push below the natural equilibrium).
- 1-D readout ≈ OpenLoop because the sustainable extent (~1.86) ≈ the per-step noise scale (~1.46) — control is trivial on the dominant controllable direction. The 2-D readout is where the closed-loop win is *visible*.
- TRACK on a slow sine is gritty because the sine amplitude is buried at the noise floor — bandwidth/noise-floor trade-off, not a controller failure.

**Integral conclusion (M3, M4, M5).** LQGI ≈ LQG everywhere on the Brain — integral does *not* help because the residual error is noise-limited at every probe budget, leaving no feedforward bias for the integrator to absorb. The mechanism *works* on a deliberately mismatched synthetic (614× error reduction); the Brain just isn't in that regime.

**See [`results/RESULTS.md`](results/RESULTS.md)** for the full milestone-by-milestone log (M0 cleanup → M6 deliverable), including the basis-free benchmarking numbers, the seed-1 × output_ssi G-direction diagnosis, the data-efficiency curve, and every saved figure with its caption.